In [1]:
%matplotlib inline

import sys, os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec

warnings.filterwarnings('ignore')
sys.path.insert(0, '/Users/dvm/Documents/DvM')

from open_dvm.analysis import CTF
from open_dvm.support.FolderStructure import FolderStructure
from open_dvm.visualization.plot import plot_ctf_timecourse

print("✓ Imports successful")

✓ Imports successful


In [2]:
# ============================================
# Configuration: Change these for your data
# ============================================
project_folder = '/Users/dvm/Library/CloudStorage/OneDrive-VrijeUniversiteitAmsterdam/projects/openDvM'
os.chdir(project_folder)

# Subject number (1-7 in this dataset)
sj = 2

# Eye-tracking quality control
eye_dict = {
    'use_tracker': True,  # Enable eye-tracking exclusion
    'window_oi': (0, 0.3),  # Window: 0-300 ms post-stimulus
    'angle_thresh': 1,  # Threshold: 1 degree visual angle
    'viewing_dist': 70,  # Viewing distance (cm)
    'screen_res': (1920, 1080),  # Screen resolution (pixels)
    'screen_h': 29,  # Screen height (cm)
    'drift_correct': (-0.2, 0)  # Drift correction window
}


In [3]:
# Run CTF encoding for all subjects (1-7)
print('\n📊 Image location across all subjects')
for subject_id in range(1, 2):
    try:
        # Load data for this subject
        df_sj, epochs_sj = FolderStructure().load_processed_epochs(
            subject_id, 'ses_01_main', 'main', eye_dict
        )

        df_sj['display_type'] = 'homogeneous'
        df_sj.loc[((df_sj.target_presence != 'absent') | 
                  (df_sj.distractor_presence != 'absent')) & 
                  (df_sj.block_type == 'main')
                  , 'display_type'] =  'heterogeneous'


        ctf_o = CTF(
            sj=subject_id, epochs=epochs_sj, df=df_sj, to_decode='img_loc',
            nr_bins=8, nr_chans=8, elec_oi='all', filter=8,
            avg_ch=True, baseline=(-0.2, 0), downsample=128,
            shift_bins = df_sj.high_prob_dist.iloc[0]
        )

        ctf_o.spatial_ctf(
            pos_labels='all', cnds = dict(block_type=[['localizer'], ['main']]),
            window_oi=(-0.2, 0.5), freqs='broadband',
            excl_factor = dict(display_type = ['heterogeneous'])
        )

        
        print(f'  ✓ Subject {subject_id} complete')
    except Exception as e:
        print(f'  ✗ Subject {subject_id} failed: {str(e)}')

print('\n✓ All subjects processed')


📊 Image location across all subjects
Reading /Users/dvm/Library/CloudStorage/OneDrive-VrijeUniversiteitAmsterdam/projects/openDvM/eeg/processed/sub_01_ses_01_main-epo.fif ...
    Found the data of interest:
        t =    -699.22 ...    1000.00 ms
        0 CTF compensation matrices available
Adding metadata with 19 columns
2891 matching events found
No baseline correction applied
0 projection items activated
Eye channel is not specified in eyedict, using HEOG as default
42 trials missing eyetracking
data (used eog instead)
Eye exclusion info saved in preprocessing file (session 1)
Creating bassiset with sin_power  7
Dropped 1425 trials after specifying excl_factor
NOTE DROPPING IS DONE IN PLACE. PLEASE REREAD DATA IF THAT CONDITION IS NECESSARY AGAIN
Setting up low-pass filter at 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 5

In [ ]:
ctf_o = CTF(
    sj=subject_id, epochs=epochs_sj, df=df_sj, to_decode='img_loc',
    nr_bins=8, nr_chans=8, elec_oi='all', filter=8,
    avg_ch=True, baseline=(-0.2, 0), downsample=128,
    shift_bins = df_sj.high_prob_dist.iloc[0]
)

### THE PROBLEM IS THAT in absent displays there is no target loc!!!!!!!
#### How best to deal with this??
ctf_o.spatial_ctf(
    pos_labels='all', cnds=dict(block_type=[['localizer'], ['main']]),
    window_oi=(-0.2, 0.5), freqs='broadband',
    excl_factor = dict(display_type = ['heterogeneous'])
)

Creating bassiset with sin_power  7
Dropped 1425 trials after specifying excl_factor
NOTE DROPPING IS DONE IN PLACE. PLEASE REREAD DATA IF THAT CONDITION IS NECESSARY AGAIN
Setting up low-pass filter at 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 845 samples (1.650 s)

Frequency 1 out of 1
Applying baseline correction (mode: mean)
Running ctf for localizer_main condition


ValueError: min() iterable argument is empty